# Vertex

### Setup & imports

In [ ]:
# Setup & imports
import os

credential_path = r""
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credential_path

In [3]:
# Setup & imports
import typing
import IPython.display
from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps

def display_image(
    image,
    max_width: int = 600,
    max_height: int = 350,
) -> None:
    pil_image = typing.cast(PIL_Image.Image, image._pil_image)
    if pil_image.mode != "RGB":
        # RGB is supported by all Jupyter environments (e.g. RGBA is not yet)
        pil_image = pil_image.convert("RGB")
    image_width, image_height = pil_image.size
    if max_width < image_width or max_height < image_height:
        # Resize to display a smaller notebook image
        pil_image = PIL_ImageOps.contain(pil_image, (max_width, max_height))
    IPython.display.display(pil_image)

In [ ]:
# Setup & imports
from vertexai.preview.vision_models import ImageGenerationModel
import vertexai

vertexai.init(project="triple-reef-456508-u6", location="us-central1")

generation_model = ImageGenerationModel.from_pretrained("imagen-4.0-generate-preview-05-20")

def imgen_api(prompt):
    return generation_model.generate_images(
        prompt="A medium-shot, cameraphone quality (2021). " + prompt + " Photo is soft focus, shot on iPhone 12, for instant messaging in 2021 (if appropriate, produce selfie style).",
        number_of_images=1,
        aspect_ratio="1:1",
        negative_prompt="",
        person_generation="",
        safety_filter_level="",
        add_watermark=True,
    )

#display_image(images[0])

In [ ]:
# Setup & imports
from pathlib import Path
import re
import ntpath
import time

QAs = []
gen = os.walk(r".\augmented\conversations")
next(gen)
for x in gen:
    directory = Path(x[0])
    text_files = list(directory.rglob('*.txt'))
    for file in text_files:
        dir = os.path.join(r".\augmented\imgen_images", file.parent.absolute().name)
        Path(dir).mkdir(parents=True, exist_ok=True)
        image_path = os.path.join(dir, ntpath.split(file.name)[1])[:-4]+".jpg"
        if not os.path.exists(image_path):
            with open(file, encoding="utf8") as file:
                lines = [line.rstrip() for line in file]
                target_line = [s for s in lines if ": Image: [" in s]
                if len(target_line) > 0:
                    match = re.search(r"\[[A-Za-z\s,-—.:\"\'\’!?\(\)&“”]+\]", target_line[0])
                    retries = 3
                    for attempt in range(retries):
                        try:
                            images = imgen_api(match.group(0)[1:-1])
                            images[0].save(image_path)
                            break
                        except Exception as e:
                            print(f"Attempt {attempt + 1} failed: {e}")
                            if attempt < retries - 1:
                                time.sleep(60)
                            else:
                                print("All attempts failed.")
                                with open(os.path.join(dir, ntpath.split(file.name)[1]), "w") as handler:
                                    handler.write("no image generated")                                         
                        
                